# 🧪 W4-D4 概念实验：图遍历 vs 向量检索，谁答得了"关系型问题"？

> 配套阅读：`ima/第4周-Day4-GraphRAG与知识图谱增强.md`（知识图谱三要素、构建流程、与 RAG 的融合在那边）
>
> 向量检索擅长"找相似的文本"，但对**多跳关系问题**（"A 用了的原料会不会引起 B？"）天然残缺。
> 这个 notebook 用一个手搭的糖水店知识图谱回答：
> 1. 三元组 → 邻接表 → 多跳 BFS 遍历，2 跳就能回答"过敏原排查"；
> 2. 同一个问题扔给向量检索，看它拿回的是什么、缺的是什么；
> 3. 把图谱画出来，亲眼看看"答案路径"长什么样。
>
> 实验环境：纯 Python 标准库（collections）+ matplotlib 画图，无图数据库。

## 实验 1：三元组 → 邻接表 → 多跳遍历

知识图谱的最小单元是三元组 `(头实体, 关系, 尾实体)`。
把它们装进邻接表，BFS 就能做"沿关系走 N 步"的查询——这是所有 GraphRAG 的引擎核心。

In [ ]:
from collections import defaultdict, deque

# 糖水店知识三元组：(头实体, 关系, 尾实体)
TRIPLES = [
    ("杨枝甘露", "含原料", "炼乳"), ("杨枝甘露", "含原料", "芒果"), ("杨枝甘露", "含原料", "西柚"),
    ("芒果双皮奶", "含原料", "水牛奶"), ("芒果双皮奶", "含原料", "芒果"),
    ("双皮奶", "含原料", "水牛奶"),
    ("芋泥波波冰", "含原料", "芋头"), ("芋泥波波冰", "含原料", "黑糖"),
    ("西瓜冰", "含原料", "西瓜"),
    ("炼乳", "属于", "乳制品"), ("水牛奶", "属于", "乳制品"), ("椰浆", "属于", "乳制品"),
    ("乳制品", "含过敏原", "乳糖蛋白"), ("芒果", "含过敏原", "漆酚类蛋白"),
]

adj = defaultdict(list)          # 正向邻接表：实体 → [(关系, 邻居), ...]
for h, r, t in TRIPLES:
    adj[h].append((r, t))

def k_hop(start, hops, rels=None):
    """从 start 出发走 hops 步（可选：只沿指定关系），返回 {终点: [路径]}"""
    reached = {start: [[start]]}
    frontier = deque([start])
    for _ in range(hops):
        nxt = {}
        for e in frontier:
            for rel, nb in adj[e]:
                if rels and rel not in rels:
                    continue
                for path in reached[e]:
                    p = path + [f"--{rel}-->", nb]
                    nxt.setdefault(nb, []).append(p)
        for e, ps in nxt.items():
            if e not in reached:
                reached[e] = ps
        frontier = deque(nxt)
    return {e: ps for e, ps in reached.items() if e != start}

print("问题：杨枝甘露 1 跳能到哪些原料？")
for e, ps in k_hop("杨枝甘露", 1, {"含原料"}).items():
    print(f"   {ps[0][0]} --含原料--> {e}")

print("\n问题：杨枝甘露 2 跳涉及哪些【过敏原】？（原料→属于/含过敏原→…）")
allergens = k_hop("杨枝甘露", 2, {"含原料", "属于", "含过敏原"})
for e, ps in allergens.items():
    if "过敏原" in ps[0][-2]:
        print(f"   路径：{' '.join(ps[0])}")

print("\n→ 2 跳就串起了『产品→原料→类别→过敏原』这条链，向量检索要做到同样的事，")
print("  需要恰好有一段文本把这个结论完整写下来——通常没有。")

## 实验 2：同一个问题，向量检索拿回了什么？

把**每条三元组渲染成一句话**作为 chunk，构建一个"文本化知识库"，
用 W4-D2 的 bigram 余弦检索问同样的问题："**哪些产品不含乳制品**"。

观察：检索确实能找回"炼乳属于乳制品"这类**碎片**，但它无法完成"全体产品 − 含乳产品"这个**全局集合运算**——
这就是 md 里说的"传统 RAG 的关系盲区"。

In [ ]:
import numpy as np

# 把三元组渲染成文本 chunk（模拟 GraphRAG 之外的另一条路线：纯文本 RAG）
chunks = {f"T{i}": f"{h}{r}{t}" for i, (h, r, t) in enumerate(TRIPLES)}
chunks["T99"] = "本店所有产品均现场制作，部分产品可去冰去糖。"

def bigrams(t):
    return [t[i:i+2] for i in range(len(t) - 1)]

def search(query, k=3):
    vocab = sorted({g for c in chunks.values() for g in bigrams(c)})
    gidx = {g: i for i, g in enumerate(vocab)}
    def vec(t):
        v = np.zeros(len(vocab))
        for g in bigrams(t):
            if g in gidx:
                v[gidx[g]] += 1
        return v
    q = vec(query); qn = q / (np.linalg.norm(q) + 1e-9)
    sims = {k_: float(qn @ (vec(c) / (np.linalg.norm(vec(c)) + 1e-9))) for k_, c in chunks.items()}
    return sorted(sims.items(), key=lambda kv: -kv[1])[:k]

q = "哪些产品不含乳制品"
print(f"查询：「{q}」→ 向量检索 Top3：")
for k_, s in search(q):
    print(f"   {s:.3f}  {chunks[k_]}")

print("\n检索给回的是『炼乳属于乳制品』『水牛奶属于乳制品』这类关系碎片，")
print("没有任何一个 chunk 列出了全部产品名单 → LLM 拿到碎片也做不出完整的排除清单。")

# ---- 同一个问题，图遍历怎么做 ----
def contains_milk(product):
    """图查询：产品 2 跳内是否能走到『乳制品』"""
    for e, ps in k_hop(product, 2, {"含原料", "属于"}).items():
        if e == "乳制品":
            return True
    return False

products = [h for h, r, t in TRIPLES if r == "含原料"]
safe = [p for p in sorted(set(products)) if not contains_milk(p)]
risky = [p for p in sorted(set(products)) if contains_milk(p)]
print(f"\n图遍历答案：含乳制品（需提示过敏顾客）= {risky}")
print(f"            不含乳制品（可选）　　　　　 = {safe}")
print("→ 图上一次反向遍历 = 完整、可解释、零遗漏的排查清单。")

## 实验 3：把图谱和"答案路径"画出来

手工布局节点位置，画出糖水店知识图谱；
红色高亮 = 实验 1 中"杨枝甘露 → 过敏原"的 2 跳答案路径——GraphRAG 的"可解释性"就是这条看得见的链。

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import font_manager

font_path = "/usr/share/fonts/opentype/noto/NotoSansCJK-Regular.ttc"
font_manager.fontManager.addfont(font_path)
font_name = font_manager.FontProperties(fname=font_path).get_name()
plt.rcParams["font.family"] = font_name
plt.rcParams["axes.unicode_minus"] = False

pos = {
    "杨枝甘露": (0.5, 4.6), "芒果双皮奶": (1.8, 4.6), "双皮奶": (3.1, 4.6),
    "芋泥波波冰": (4.4, 4.6), "西瓜冰": (5.7, 4.6),
    "炼乳": (0.2, 3.2), "芒果": (1.1, 3.2), "西柚": (2.0, 3.2), "水牛奶": (2.9, 3.2),
    "芋头": (3.8, 3.2), "黑糖": (4.6, 3.2), "西瓜": (5.5, 3.2),
    "乳制品": (1.6, 1.8), "椰浆": (0.3, 2.5), "漆酚类蛋白": (3.0, 0.6), "乳糖蛋白": (1.0, 0.6),
}
answer_path = {("杨枝甘露", "芒果"), ("芒果", "漆酚类蛋白")}

fig, ax = plt.subplots(figsize=(9, 6.5))
for h, r, t in TRIPLES:
    x1, y1 = pos[h]; x2, y2 = pos[t]
    hot = (h, t) in answer_path
    ax.annotate("", xy=(x2, y2), xytext=(x1, y1),
                arrowprops=dict(arrowstyle="->", color="#c0504d" if hot else "#999",
                                lw=2.5 if hot else 1.0, alpha=1 if hot else 0.6))
    ax.text((x1 + x2) / 2, (y1 + y2) / 2 + 0.08, r, fontsize=7.5,
            ha="center", color="#c0504d" if hot else "#777")

for n, (x, y) in pos.items():
    kind = "产品" if y > 4 else ("原料" if y > 2.5 else "类别/过敏原")
    color = {"产品": "#4f81bd", "原料": "#9bbb59", "类别/过敏原": "#c0504d"}[kind]
    size = 300 + 80 * sum(1 for h, _, t in TRIPLES if h == n or t == n)
    hot = n in {"杨枝甘露", "芒果", "漆酚类蛋白"}
    ax.scatter([x], [y], s=size * (1.6 if hot else 1), color=color,
               edgecolors="#c0504d" if hot else "white", linewidths=2 if hot else 1, zorder=3)
    ax.text(x, y, n, ha="center", va="center", fontsize=8.5,
            color="white" if hot else "black", zorder=4)

ax.set_xlim(-0.5, 6.4); ax.set_ylim(0.2, 5.2)
ax.axis("off")
ax.set_title("糖水店知识图谱（蓝=产品 绿=原料 红=类别/过敏原）\n红色路径：杨枝甘露 --含原料--> 芒果 --含过敏原--> 漆酚类蛋白（实验1的答案）")
plt.tight_layout(); plt.show()

print("复盘：向量检索返回『相似的句子』，图遍历返回『沿关系走的路径』；")
print("     关系型/多跳型问题（溯源、排查、推荐）只有后者能给出完整且可解释的答案。")

## 结论

| 问题 | 实验证据 |
|---|---|
| 图谱怎么用 | 实验1：三元组→邻接表→2跳BFS 回答过敏原排查 |
| 向量检索的盲区 | 实验2：只能找回关系碎片，做不了"全体产品−含乳产品"的全局运算 |
| 为什么可解释 | 实验3：答案就是图上一条高亮路径，每一步都有关系依据 |

→ 深入阅读：`ima/第4周-Day4-GraphRAG与知识图谱增强.md`（三元组抽取、GraphRAG 工作流、与向量 RAG 的融合策略、4 个误区）